

## Objective
This notebook is used to train the stage-2 ranking model (XGBoost-based) using engineered user-item-context features.

## Data Sources
Key datasets/artifacts used in this notebook:
- candidate pool/features
- label/interaction outcomes
- customer and item feature artifacts

## Core Workflow
- Load training data with labels and ranking features.
- Perform feature filtering, transformation, and train/validation split.
- Train XGBoost ranker/classifier configuration.
- Evaluate with ranking/quality metrics and diagnostics.
- Save final model and feature metadata for inference.

## Expected Outputs
- Trained XGBoost ranking model artifact.
- Feature importance/metric outputs for model monitoring.
- Serialized files needed for production scoring.

## Role in Recommendation Pipeline
Produces final ranking logic that orders candidates generated in stage-1.



<!-- AUTO_CELL_EXPLANATION -->
### Cell 1: # Run only if xgboost is not installed
**Explanation:** This cell trains or fits model components in the ranker training and XGBoost pipeline.

**Possible Output:** Possible output: console/text output.


In [1]:
# Run only if xgboost is not installed
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable


<!-- AUTO_CELL_EXPLANATION -->
### Cell 2: import ast
**Explanation:** This cell imports required dependencies for the ranker training and XGBoost pipeline.
## This code initializes a recommendation system ranking pipeline.
- It defines file paths for multiple data sources such as basket history, item catalog, category rules, and NGCF candidate outputs.
- Creates an output directory to store trained models and results.
- Sets random seeds to ensure reproducibility.
- Imports required libraries (NumPy, Pandas, Scikit-learn, XGBoost).
- Finally, it checks whether all required input files exist.

**Possible Output:** Possible output: console/text output.
- Basket file exists: True
- Category lookup file exists: True
- Item catalog file exists: True
- Category rule file exists: True
- NGCF candidate file exists: True
###### **Output folder:** C:\D drive\item_recommendation_model_create\data\ranker_training_output


In [2]:
import ast
import json
import pickle
import random
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

import xgboost as xgb

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

random.seed(42)
np.random.seed(42)

BASE_DIR = Path(r"C:\D drive\item_recommendation_model_create")
DATA_DIR = BASE_DIR / "data"

BASKET_EMBEDDING_DIR = DATA_DIR / "basket_embedding_output"
ITEM_CATALOG_DIR = DATA_DIR / "item_catalog_output"
NGCF_OUTPUT_DIR = DATA_DIR / "ngcf_output"

RANKER_OUTPUT_DIR = DATA_DIR / "ranker_training_output"
RANKER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASKET_FILE = BASKET_EMBEDDING_DIR / "customer_purchase_pattern_history_final.csv"
CATEGORY_LOOKUP_FILE = BASKET_EMBEDDING_DIR / "category_embedding_lookup.csv"
ITEM_CATALOG_FILE = ITEM_CATALOG_DIR / "item_catalog.csv"
CATEGORY_RULE_FILE = ITEM_CATALOG_DIR / "category_rule_artifacts.json"
NGCF_TOP_CANDIDATES_FILE = NGCF_OUTPUT_DIR / "ngcf_top_candidates.csv"

RANKER_DATASET_FILE = RANKER_OUTPUT_DIR / "ranker_training_dataset.csv"
MODEL_JSON_FILE = RANKER_OUTPUT_DIR / "xgboost_ranker_model.json"
MODEL_PICKLE_FILE = RANKER_OUTPUT_DIR / "xgboost_ranker_model.pkl"
FEATURE_COLUMNS_FILE = RANKER_OUTPUT_DIR / "ranker_feature_columns.json"
TRAINING_SUMMARY_FILE = RANKER_OUTPUT_DIR / "ranker_training_summary.json"
FEATURE_IMPORTANCE_FILE = RANKER_OUTPUT_DIR / "ranker_feature_importance.csv"
TEST_PREDICTION_FILE = RANKER_OUTPUT_DIR / "ranker_test_predictions.csv"
FINAL_RESULT_FILE = RANKER_OUTPUT_DIR / "final_result_summary.csv"

print("Basket file exists:", BASKET_FILE.exists())
print("Category lookup file exists:", CATEGORY_LOOKUP_FILE.exists())
print("Item catalog file exists:", ITEM_CATALOG_FILE.exists())
print("Category rule file exists:", CATEGORY_RULE_FILE.exists())
print("NGCF candidate file exists:", NGCF_TOP_CANDIDATES_FILE.exists())
print("Output folder:", RANKER_OUTPUT_DIR)

Basket file exists: True
Category lookup file exists: True
Item catalog file exists: True
Category rule file exists: True
NGCF candidate file exists: True
Output folder: C:\D drive\item_recommendation_model_create\data\ranker_training_output


<!-- AUTO_CELL_EXPLANATION -->
### Cell 3: basket_df = pd.read_csv(BASKET_FILE)
# Data Loading & Preprocessing – Code Overview

## 1. Explanation (Precise)
- Loads datasets: basket history, category lookup, item catalog, and NGCF candidates.
- Reads rule artifacts from JSON (category constraints and mappings).
- Uses a fallback `category_family_map` if not present in the rule file.
- Handles missing NGCF file by creating an empty DataFrame.
- Cleans column names (removes whitespace).
- Converts NGCF columns to proper data types.
- Prints dataset shapes and displays sample rows.
## 2. Possible Output
```bash
basket_df shape: (50000, 10)
category_lookup_df shape: (200, 3)
item_catalog_df shape: (10000, 8)
ngcf_candidates_df shape: (300000, 4)
category_family_map size: 45


In [3]:
basket_df = pd.read_csv(BASKET_FILE)
category_lookup_df = pd.read_csv(CATEGORY_LOOKUP_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)

with open(CATEGORY_RULE_FILE, "r", encoding="utf-8") as f:
    rule_artifacts = json.load(f)

category_allowed_map = rule_artifacts["category_allowed_map"]
all_valid_categories = rule_artifacts["all_valid_categories"]

# Category family map is required for dynamic mixed basket ranking.
# If the rule artifact does not contain it yet, this fallback keeps the notebook runnable.
default_category_family_map = {
    "Bakery-Bread": "breakfast",
    "Beverage-Hot": "breakfast",
    "Dairy-Milk": "breakfast",
    "Dairy-Other": "breakfast",
    "Protein-Egg": "breakfast",

    "Meat-Fresh": "cooking",
    "Meat-Processed": "processed_food",
    "Fish-Fresh": "cooking",
    "Pantry-Rice": "cooking",
    "Pantry-Oils": "cooking",
    "Pantry-Pulses": "cooking",
    "Pantry-Flour": "cooking",
    "Pantry-Grains": "cooking",
    "Pantry-Sweeteners": "dessert",
    "pantry salt": "cooking",
    "Spices-Cooking": "cooking",
    "sauce item": "cooking",
    "Veg-Cooking": "cooking",
    "general_cooking_vegetables": "cooking",
    "Pickle": "cooking",

    "Snacks-General": "snack",
    "Chocolates-Sweets": "snack",
    "Beverage-Carbonated": "snack",
    "Beverage-Juice": "snack",
    "Beverage-Water": "beverage",
    "noodles_pasta_and_haleem": "snack",

    "Desserts-Traditional": "dessert",
    "Dry-Fruits": "dessert",
    "Fruits-Fresh": "dessert",

    "Household-AirCare": "household",
    "Household-Cleaning": "household",
    "Household-Kitchen": "household",
    "Household-Laundry": "household",
    "Household-Utility": "household",
    "household hygene": "household",

    "Personal-Care-Bath": "personal_care",
    "Personal-Care-Cosmetics": "personal_care",
    "Personal-Care-Hair": "personal_care",
    "Personal-Care-Oral": "personal_care",
    "Personal-Care-Sanitary": "personal_care",

    "cleaning_clothes": "clothing",
    "clothing - male": "clothing",
    "clothing accessories female": "clothing",
    "clothing accessories male": "clothing",
    "clothing babies": "clothing",
    "clothing female": "clothing"
}

category_family_map = rule_artifacts.get("category_family_map", default_category_family_map)

if not category_family_map:
    category_family_map = default_category_family_map

if NGCF_TOP_CANDIDATES_FILE.exists():
    ngcf_candidates_df = pd.read_csv(NGCF_TOP_CANDIDATES_FILE)
else:
    ngcf_candidates_df = pd.DataFrame(
        columns=["customerId", "itemId", "ngcf_score", "ngcf_rank"]
    )

basket_df.columns = [c.strip() for c in basket_df.columns]
category_lookup_df.columns = [c.strip() for c in category_lookup_df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]
ngcf_candidates_df.columns = [c.strip() for c in ngcf_candidates_df.columns]

if not ngcf_candidates_df.empty:
    ngcf_candidates_df["customerId"] = ngcf_candidates_df["customerId"].astype(int)
    ngcf_candidates_df["itemId"] = ngcf_candidates_df["itemId"].astype(int)
    ngcf_candidates_df["ngcf_score"] = pd.to_numeric(
        ngcf_candidates_df["ngcf_score"],
        errors="coerce"
    ).fillna(0.0)
    ngcf_candidates_df["ngcf_rank"] = pd.to_numeric(
        ngcf_candidates_df["ngcf_rank"],
        errors="coerce"
    ).fillna(999).astype(int)

print("basket_df shape:", basket_df.shape)
print("category_lookup_df shape:", category_lookup_df.shape)
print("item_catalog_df shape:", item_catalog_df.shape)
print("ngcf_candidates_df shape:", ngcf_candidates_df.shape)
print("category_family_map size:", len(category_family_map))

display(basket_df.head())
display(category_lookup_df.head())
display(item_catalog_df.head())
display(ngcf_candidates_df.head())


basket_df shape: (184878, 15)
category_lookup_df shape: (46, 9)
item_catalog_df shape: (229, 11)
ngcf_candidates_df shape: (37000, 6)
category_family_map size: 46


,transactionId,customerId,purchaseDate,categorySet,itemIdSet,itemNameSet,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth,basket_category_embedding
0,1,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","{952, 453, 15262, 32441, 13986, 332, 13917, 13921, 704, 292, 14975}","{Chashi Aroma.Chinigura Rice-2kg, Saad Testy Bit Salt-100gm, Cow Brain-K, Ela vista pomace olive oil 250 ml, Cow Stomach-K, ACI Pure Salt-1Kg, Rui...",Winter,1,0,1,Afternoon,3,1,1,"[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,2,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","{13986, 12655, 332, 952, 3191, 976}","{Cow Stomach-K, Ramisa Nut(Oil)-200gm, ACI Pure Salt-1Kg, Chashi Aroma.Chinigura Rice-2kg, Frooto Mango Juice-500ml, Lata Rice-KG}",Winter,1,0,1,Afternoon,3,1,1,"[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,3,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing female}","{12637, 453, 32269, 17809, 6848, 25811}","{Ramisa-Nakshi Pitha Box-4 Pcs, Saad Testy Bit Salt-100gm, Gopal Ind Like Me Bra S-38 (4/23), Foster Clarks Corn Flour-200gm, Garlic China Big-(kg...",Winter,1,0,1,Noon,2,1,1,"[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,4,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","{2030, 13958, 14983, 453, 488, 15131, 976, 4958}","{Pran Aromatic Rice-2kg, Shrimp Fish Bagda, Cauliflower(B)-Pc, Saad Testy Bit Salt-100gm, Bacha Boot Pulse 1Kg, Atta Fruits, Lata Rice-KG, Teer Su...",Winter,1,0,1,Noon,2,1,1,"[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,5,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","{976, 332, 13986, 3612, 12826, 3191, 12834, 952}","{Lata Rice-KG, ACI Pure Salt-1Kg, Cow Stomach-K, Italiano Sweet Chilli Sauce-340g, Chia Seed (Dana) New-100g, Frooto Mango Juice-500ml, Prince Bet...",Winter,1,0,1,Night,5,1,1,"[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"


,category,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,cat_emb_8
0,Bakery-Bread,-74.173568,-111.558025,-192.203051,-70.900538,-27.881714,-73.214051,24.189736,161.398282
1,Beverage-Carbonated,-85.186907,-42.566291,242.853551,7.961231,-155.819533,0.959077,-104.045821,152.333994
2,Beverage-Hot,-28.961506,-26.255853,-7.816855,264.783572,116.117895,19.741289,-79.332576,-52.988355
3,Beverage-Juice,121.029090,179.488928,-119.588266,-205.621190,-16.090835,-78.735413,-165.299904,179.731694
4,Beverage-Water,284.556475,104.078329,-57.684418,9.785665,-68.273613,-132.973486,-38.705386,-119.340987


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0


,customerId,itemId,ngcf_score,ngcf_rank,itemName,category
0,2242,26701,0.999397,1,Nescafe Classic Jar-25gm,Beverage-Hot
1,2242,3554,0.999288,2,Parachute Skin Pure Aloe Vera Gel- 50ml,Personal-Care-Cosmetics
2,2242,7075,0.999280,3,Herman Peanut Butter-340gm,Dairy-Other
3,2242,30788,0.999278,4,Teer Atta 2kg,Pantry-Flour
4,2242,3182,0.999265,5,Starship Chocolate Milk-200ml,Dairy-Milk


<!-- AUTO_CELL_EXPLANATION -->
### Cell 4: def normalize_text(value):
# Helper Functions – Code Overview

## 1. Explanation (Precise)
- Defines utility functions for data cleaning and parsing.
- `normalize_text`: cleans and standardizes text.
- `parse_set_like`: converts string sets (e.g., "{a,b}") into lists.
- `parse_numeric_set`: extracts numeric values from set-like strings.
- `parse_vector`: safely converts string/list into NumPy vector.
- `cosine_sim`: computes cosine similarity between two vectors.
- `safe_int`: safely converts values to integer with fallback.
- These helpers ensure robust preprocessing and feature handling.
## 2. Possible Output
```bash
Helper functions ready


In [4]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    
    return value


def parse_set_like(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    if value.startswith("{") and value.endswith("}"):
        value = value[1:-1].strip()
    
    if value == "":
        return []
    
    return [x.strip() for x in value.split(",") if x.strip()]


def parse_numeric_set(value):
    values = parse_set_like(value)
    output = []
    
    for value in values:
        try:
            output.append(int(float(value)))
        except Exception:
            pass
    
    return output


def parse_vector(value):
    if pd.isna(value):
        return None
    
    try:
        if isinstance(value, str):
            value = ast.literal_eval(value)
        
        return np.array(value, dtype=float)
    
    except Exception:
        return None


def cosine_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return 0.0
    
    vec1 = np.array(vec1, dtype=float).reshape(1, -1)
    vec2 = np.array(vec2, dtype=float).reshape(1, -1)
    
    if np.allclose(vec1, 0) or np.allclose(vec2, 0):
        return 0.0
    
    return float(cosine_similarity(vec1, vec2)[0][0])


def safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


print("Helper functions ready")

Helper functions ready


<!-- AUTO_CELL_EXPLANATION -->
### Cell 5: required_basket_cols = [
# Basket Data Validation & Preparation – Code Overview

## 1. Explanation (Precise)
- Validates that all required basket columns exist.
- Raises an error if any required column is missing.
- Creates parsed list columns for item IDs, categories, and item names.
- Converts basket embedding into vector format.
- Resets index and assigns a unique `basket_id`.
- Prepares structured data for downstream modeling.

## 2. Possible Output
```bash
Prepared basket shape: (50000, 18)


In [5]:
required_basket_cols = [
    "customerId",
    "purchaseDate",
    "categorySet",
    "itemIdSet",
    "itemNameSet",
    "season",
    "seasonLabel",
    "isHoliday",
    "isFestival",
    "timeSlot",
    "timeSlotLabel",
    "monthPartLabel",
    "weekOfMonth",
    "basket_category_embedding"
]

missing_cols = [col for col in required_basket_cols if col not in basket_df.columns]

if missing_cols:
    raise ValueError(f"Missing basket columns: {missing_cols}")

basket_df = basket_df.copy()

basket_df["itemIdSet_list"] = basket_df["itemIdSet"].apply(parse_numeric_set)
basket_df["categorySet_list"] = basket_df["categorySet"].apply(parse_set_like)
basket_df["itemNameSet_list"] = basket_df["itemNameSet"].apply(parse_set_like)
basket_df["basket_category_embedding_vec"] = basket_df["basket_category_embedding"].apply(parse_vector)

basket_df = basket_df.reset_index(drop=True)
basket_df["basket_id"] = basket_df.index + 1

print("Prepared basket shape:", basket_df.shape)

display(
    basket_df[
        [
            "basket_id",
            "customerId",
            "purchaseDate",
            "itemIdSet_list",
            "categorySet_list",
            "basket_category_embedding_vec"
        ]
    ].head()
)

Prepared basket shape: (184878, 20)


,basket_id,customerId,purchaseDate,itemIdSet_list,categorySet_list,basket_category_embedding_vec
0,1,23433,2025-01-01,"[952, 453, 15262, 32441, 13986, 332, 13917, 13921, 704, 292, 14975]","[Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking]","[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,2,23553,2025-01-01,"[13986, 12655, 332, 952, 3191, 976]","[Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice]","[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,3,23453,2025-01-01,"[12637, 453, 32269, 17809, 6848, 25811]","[Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing female]","[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,4,23437,2025-01-01,"[2030, 13958, 14983, 453, 488, 15131, 976, 4958]","[Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners]","[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,5,23401,2025-01-01,"[976, 332, 13986, 3612, 12826, 3191, 12834, 952]","[Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking]","[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"


<!-- AUTO_CELL_EXPLANATION -->
### Cell 6: required_catalog_cols = ["itemId", "itemName", "category"]
# Item Catalog & Category Embedding Preparation – Code Overview

## 1. Explanation (Precise)
- Validates required catalog columns (`itemId`, `itemName`, `category`).
- Raises an error if any required column is missing.
- Cleans and standardizes item names and categories.
- Creates mappings:
  - `itemId → itemName`
  - `itemId → category`
- Extracts category embedding columns (`cat_emb_*`).
- Builds `category → embedding vector` dictionary.
- Prepares catalog and embedding data for feature engineering.
## 2. Possible Output
```bash
Total catalog items: 10000
Total category vectors: 200
Embedding columns: ['cat_emb_0', 'cat_emb_1', 'cat_emb_2', ...]


In [6]:
required_catalog_cols = ["itemId", "itemName", "category"]

missing_catalog_cols = [col for col in required_catalog_cols if col not in item_catalog_df.columns]

if missing_catalog_cols:
    raise ValueError(f"Missing catalog columns: {missing_catalog_cols}")

item_catalog_df = item_catalog_df.copy()

item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["itemName"] = item_catalog_df["itemName"].apply(normalize_text)
item_catalog_df["category"] = item_catalog_df["category"].apply(normalize_text)

item_to_name = dict(zip(item_catalog_df["itemId"], item_catalog_df["itemName"]))
item_to_category = dict(zip(item_catalog_df["itemId"], item_catalog_df["category"]))

embedding_cols = [col for col in category_lookup_df.columns if col.startswith("cat_emb_")]

if not embedding_cols:
    raise ValueError("category_embedding_lookup.csv must contain cat_emb columns")

category_lookup_df["category"] = category_lookup_df["category"].apply(normalize_text)

category_to_vector = {}

for _, row in category_lookup_df.iterrows():
    category = row["category"]
    vector = row[embedding_cols].values.astype(float)
    category_to_vector[category] = vector

print("Total catalog items:", len(item_to_category))
print("Total category vectors:", len(category_to_vector))
print("Embedding columns:", embedding_cols)

display(item_catalog_df.head())

Total catalog items: 229
Total category vectors: 46
Embedding columns: ['cat_emb_1', 'cat_emb_2', 'cat_emb_3', 'cat_emb_4', 'cat_emb_5', 'cat_emb_6', 'cat_emb_7', 'cat_emb_8']


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 7: exploded_rows = []
# Basket Explosion (Item-Level Transformation) – Code Overview

## 1. Explanation (Precise)
- Converts basket-level data into **item-level rows**.
- Iterates through each basket and expands `itemIdSet_list`.
- Each item becomes a separate row with basket and contextual features.
- Maps `itemId` to `itemName` and `category`.
- Preserves time, season, and event-related features.
- Creates a structured dataset for model training.

## 2. Possible Output
```bash
Exploded shape: (150000, 14)


In [7]:
exploded_rows = []

for _, row in basket_df.iterrows():
    basket_id = int(row["basket_id"])
    customer_id = int(row["customerId"])
    
    for item_id in row["itemIdSet_list"]:
        item_id = int(item_id)
        
        exploded_rows.append({
            "basket_id": basket_id,
            "customerId": customer_id,
            "purchaseDate": row["purchaseDate"],
            "itemId": item_id,
            "itemName": item_to_name.get(item_id, ""),
            "category": item_to_category.get(item_id, ""),
            "season": row["season"],
            "seasonLabel": safe_int(row["seasonLabel"]),
            "isHoliday": safe_int(row["isHoliday"]),
            "isFestival": safe_int(row["isFestival"]),
            "timeSlot": row["timeSlot"],
            "timeSlotLabel": safe_int(row["timeSlotLabel"]),
            "monthPartLabel": safe_int(row["monthPartLabel"]),
            "weekOfMonth": safe_int(row["weekOfMonth"])
        })

exploded_df = pd.DataFrame(exploded_rows)

print("Exploded shape:", exploded_df.shape)
display(exploded_df.head())

Exploded shape: (1000000, 14)


,basket_id,customerId,purchaseDate,itemId,itemName,category,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth
0,1,23433,2025-01-01,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,Winter,1,0,1,Afternoon,3,1,1
1,1,23433,2025-01-01,453,Saad Testy Bit Salt-100gm,pantry salt,Winter,1,0,1,Afternoon,3,1,1
2,1,23433,2025-01-01,15262,Cow Brain-K,Meat-Fresh,Winter,1,0,1,Afternoon,3,1,1
3,1,23433,2025-01-01,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,Winter,1,0,1,Afternoon,3,1,1
4,1,23433,2025-01-01,13986,Cow Stomach-K,Meat-Fresh,Winter,1,0,1,Afternoon,3,1,1


<!-- AUTO_CELL_EXPLANATION -->
### Cell 8: item_pair_counter = Counter()
# 📈 Co-occurrence & Context Feature Engineering – Code Overview

## 1. Explanation (Precise)
- Builds multiple **frequency-based features** from exploded item-level data.
- Computes:
  - Item–item co-occurrence (`item_pair_counter`)
  - Context-aware item frequency (season, holiday, time, etc.)
  - User–item interaction counts
  - User–category preferences
  - Overall category popularity
- Creates lookup structures for fast retrieval:
  - `pair_to_related_items`
  - `context_to_items`
  - `user_to_items`
- These features are later used for recommendation ranking.

## 2. Possible Output
```bash
Item pair counter: 250000
Context item counter: 180000
User item counter: 120000
User category counter: 80000
Context lookup: 5000
User lookup: 10000

In [ ]:
item_pair_counter = Counter()
context_item_counter = Counter()
user_item_counter = Counter()
user_category_counter = Counter()
category_popularity_counter = Counter()

pair_to_related_items = defaultdict(Counter)
context_to_items = defaultdict(Counter)
user_to_items = defaultdict(Counter)

for basket_id, group in exploded_df.groupby("basket_id", sort=False):
    items = list(dict.fromkeys(group["itemId"].astype(int).tolist()))
    
    customer_id = int(group["customerId"].iloc[0])
    
    for item_id in items:
        item_rows = group[group["itemId"].astype(int) == int(item_id)]
        
        if item_rows.empty:
            continue
        
        category = item_rows["category"].iloc[0]
        user_to_items[customer_id][int(item_id)] += 1
    
    for i in range(len(items)):
        item_i = int(items[i])
        
        for j in range(len(items)):
            if i == j:
                continue
            
            item_j = int(items[j])
            
            item_pair_counter[(item_i, item_j)] += 1
            pair_to_related_items[item_j][item_i] += 1

for _, row in exploded_df.iterrows():
    context_key = (
        row["season"],
        int(row["isHoliday"]),
        int(row["isFestival"]),
        row["timeSlot"],
        int(row["weekOfMonth"])
    )
    
    item_id = int(row["itemId"])
    customer_id = int(row["customerId"])
    category = row["category"]
    
    context_item_counter[(context_key, item_id)] += 1
    context_to_items[context_key][item_id] += 1
    
    user_item_counter[(customer_id, item_id)] += 1
    user_category_counter[(customer_id, category)] += 1
    category_popularity_counter[category] += 1

print("Item pair counter:", len(item_pair_counter))
print("Context item counter:", len(context_item_counter))
print("User item counter:", len(user_item_counter))
print("User category counter:", len(user_category_counter))
print("Context lookup:", len(context_to_items))
print("User lookup:", len(user_to_items))

<!-- AUTO_CELL_EXPLANATION -->
### Cell 9: ngcf_candidate_lookup = defaultdict(list)
# NGCF Candidate Lookup & Utilities – Code Overview

## 1. Explanation (Precise)
- Builds fast lookup structures from NGCF output:
  - `customer → candidate items`
  - `(customer, item) → score`
  - `(customer, item) → rank`
- Sorts candidates by rank for each customer.
- Provides helper functions:
  - `get_top_ngcf_candidates`: returns top-N items per user
  - `get_ngcf_score`: retrieves NGCF score
  - `get_ngcf_rank`: retrieves ranking position
- Uses default values if data is missing (score = 0.0, rank = 999).

## 2. Possible Output

```bash
NGCF lookup ready
Customers with NGCF candidates: 10000

In [ ]:
ngcf_candidate_lookup = defaultdict(list)
ngcf_score_lookup = {}
ngcf_rank_lookup = {}

if not ngcf_candidates_df.empty:
    ngcf_candidates_df = ngcf_candidates_df.sort_values(["customerId", "ngcf_rank"])
    
    for _, row in ngcf_candidates_df.iterrows():
        customer_id = int(row["customerId"])
        item_id = int(row["itemId"])
        score = float(row["ngcf_score"])
        rank = int(row["ngcf_rank"])
        
        ngcf_candidate_lookup[customer_id].append(item_id)
        ngcf_score_lookup[(customer_id, item_id)] = score
        ngcf_rank_lookup[(customer_id, item_id)] = rank


def get_top_ngcf_candidates(customer_id, top_n=80):
    customer_id = int(customer_id)
    return ngcf_candidate_lookup.get(customer_id, [])[:top_n]


def get_ngcf_score(customer_id, candidate_item_id):
    return float(
        ngcf_score_lookup.get(
            (int(customer_id), int(candidate_item_id)),
            0.0
        )
    )


def get_ngcf_rank(customer_id, candidate_item_id):
    return int(
        ngcf_rank_lookup.get(
            (int(customer_id), int(candidate_item_id)),
            999
        )
    )


print("NGCF lookup ready")
print("Customers with NGCF candidates:", len(ngcf_candidate_lookup))

<!-- AUTO_CELL_EXPLANATION -->
### Cell 10: household_group = [
# Dynamic Category Rules & Filtering – Code Overview

## 1. Explanation (Precise)
- Defines category groups (household, personal care, clothing).
- Builds a **category rule cache** for fast lookup based on time slots.
- Maps each category to a **category family**.
- Generates **dynamic category weights** based on:
  - Basket context
  - Time slot
  - Rule-based boosts (e.g., meat → cooking items, bread → breakfast items)
- Produces allowed categories dynamically for recommendation.
- Applies filtering on item catalog based on allowed categories.
- Supports strict filtering (exclude same categories in some cases).


## 2. Possible Output

```bash
Dynamic mixed basket category rule functions ready


In [ ]:
household_group = [
    "Household-AirCare",
    "Household-Cleaning",
    "Household-Kitchen",
    "Household-Laundry",
    "Household-Utility",
    "household hygene"
]

personal_care_group = [
    "Personal-Care-Bath",
    "Personal-Care-Cosmetics",
    "Personal-Care-Hair",
    "Personal-Care-Oral",
    "Personal-Care-Sanitary"
]

clothing_group = [
    "cleaning_clothes",
    "clothing - male",
    "clothing accessories female",
    "clothing accessories male",
    "clothing babies",
    "clothing female"
]

category_allowed_cache = {}

for source_category, rule in category_allowed_map.items():
    source_category = str(source_category).strip()
    any_allowed = rule.get("Any", [])

    for time_slot in ["Morning", "Noon", "Afternoon", "Evening", "Night"]:
        if time_slot in rule:
            category_allowed_cache[(source_category, time_slot)] = list(rule[time_slot])
        else:
            category_allowed_cache[(source_category, time_slot)] = list(any_allowed)

    category_allowed_cache[(source_category, None)] = list(any_allowed)


def get_category_family(category):
    return category_family_map.get(str(category).strip(), "other")


def get_rule_allowed_categories(source_category, time_slot=None):
    source_category = str(source_category).strip()

    if (source_category, time_slot) in category_allowed_cache:
        return category_allowed_cache[(source_category, time_slot)]

    if (source_category, None) in category_allowed_cache:
        return category_allowed_cache[(source_category, None)]

    return []


def build_dynamic_category_weights(context_categories, time_slot=None):
    context_categories = [
        str(c).strip() for c in context_categories
        if str(c).strip()
    ]
    context_categories = list(dict.fromkeys(context_categories))

    allowed_weight_counter = Counter()
    source_family_counter = Counter()

    for source_category in context_categories:
        source_family = get_category_family(source_category)
        source_family_counter[source_family] += 1

        source_allowed = get_rule_allowed_categories(
            source_category,
            time_slot
        )

        if not source_allowed:
            continue

        for rank, target_category in enumerate(source_allowed, start=1):
            target_family = get_category_family(target_category)
            base_weight = 1.0 / rank

            if target_family == source_family:
                base_weight += 0.45

            if source_category in ["Meat-Fresh", "Fish-Fresh"]:
                if target_category in [
                    "pantry salt",
                    "Pantry-Oils",
                    "general_cooking_vegetables",
                    "Pantry-Rice",
                    "Spices-Cooking",
                    "sauce item"
                ]:
                    base_weight += 0.65

            if source_category == "Bakery-Bread":
                if time_slot == "Morning":
                    if target_category in [
                        "Beverage-Hot",
                        "Dairy-Milk",
                        "Dairy-Other",
                        "Protein-Egg"
                    ]:
                        base_weight += 0.60
                else:
                    if target_category in [
                        "Snacks-General",
                        "Meat-Processed",
                        "sauce item",
                        "noodles_pasta_and_haleem"
                    ]:
                        base_weight += 0.45

            if source_category in [
                "clothing - male",
                "clothing female",
                "clothing babies"
            ]:
                if target_family == "clothing":
                    base_weight += 0.70

            allowed_weight_counter[target_category] += base_weight

    return allowed_weight_counter, source_family_counter


def build_allowed_categories_from_context(context_categories, time_slot=None):
    context_categories = [
        str(c).strip() for c in context_categories
        if str(c).strip()
    ]
    context_categories = list(dict.fromkeys(context_categories))
    context_category_set = set(context_categories)

    allowed_weight_counter, source_family_counter = build_dynamic_category_weights(
        context_categories=context_categories,
        time_slot=time_slot
    )

    allowed_categories = [
        category for category, _ in allowed_weight_counter.most_common()
    ]

    strict_different_category_only = False

    if "Fish-Fresh" in context_category_set or "Meat-Fresh" in context_category_set:
        strict_different_category_only = True

    if strict_different_category_only:
        allowed_categories = [
            category for category in allowed_categories
            if category not in context_category_set
        ]

    if not allowed_categories:
        for source_category in context_categories:
            source_allowed = get_rule_allowed_categories(
                source_category,
                time_slot
            )
            allowed_categories.extend(source_allowed)

        allowed_categories = list(dict.fromkeys(allowed_categories))

    if not allowed_categories:
        allowed_categories = item_catalog_df["category"].dropna().unique().tolist()

    rule_mode = "mixed_basket" if len(context_categories) > 1 else (
        context_categories[0] if context_categories else "fallback_all"
    )

    return allowed_categories, strict_different_category_only, rule_mode, dict(allowed_weight_counter)


catalog_by_category = {
    category: group.copy()
    for category, group in item_catalog_df.groupby("category", sort=False)
}


def filter_catalog_by_allowed_categories(
    allowed_categories,
    input_categories=None,
    strict_different_category_only=False
):
    if input_categories is None:
        input_categories = []

    frames = []

    for category in allowed_categories:
        if category in catalog_by_category:
            frames.append(catalog_by_category[category])

    if not frames:
        return item_catalog_df.iloc[0:0].copy()

    temp = pd.concat(frames, ignore_index=True)

    if strict_different_category_only:
        input_category_set = set(input_categories)
        temp = temp[~temp["category"].isin(input_category_set)].copy()

    return temp


print("Dynamic mixed basket category rule functions ready")


<!-- AUTO_CELL_EXPLANATION -->
### Cell 11: def get_item_cooccurrence_score(candidate_item_id, context_item_ids):
# Feature Engineering Helper Functions – Code Overview

## 1. Explanation (Precise)
- Defines scoring functions used to generate **ranking features**.
- Computes:
  - Item co-occurrence score (based on basket history)
  - Customer–item interaction score
  - Customer–category affinity
  - Context-based popularity (season, time, etc.)
  - Embedding-based similarity (category vectors)
  - Global category popularity
- Builds a **context embedding vector** from items in the basket.
- These features are later used for model training (e.g., XGBoost ranker).
## 2. Possible Output
```bash
Feature helper functions ready


In [ ]:
def get_item_cooccurrence_score(candidate_item_id, context_item_ids):
    if len(context_item_ids) == 0:
        return 0.0
    
    scores = []
    
    for context_item_id in context_item_ids:
        scores.append(item_pair_counter.get((int(candidate_item_id), int(context_item_id)), 0))
    
    return float(np.mean(scores)) if scores else 0.0


def get_customer_history_score(customer_id, candidate_item_id):
    return float(user_item_counter.get((int(customer_id), int(candidate_item_id)), 0))


def get_category_affinity_score(customer_id, candidate_category):
    return float(user_category_counter.get((int(customer_id), candidate_category), 0))


def get_context_popularity_score(context_key, candidate_item_id):
    return float(context_item_counter.get((context_key, int(candidate_item_id)), 0))


def build_context_embedding_from_items(context_item_ids):
    vectors = []
    
    for item_id in context_item_ids:
        category = item_to_category.get(int(item_id), "")
        
        if category in category_to_vector:
            vectors.append(category_to_vector[category])
    
    if len(vectors) == 0:
        return None
    
    return np.mean(vectors, axis=0)


def get_embedding_similarity_score(context_vector, candidate_item_id):
    candidate_category = item_to_category.get(int(candidate_item_id), "")
    candidate_vector = category_to_vector.get(candidate_category, None)
    
    return cosine_sim(context_vector, candidate_vector)


def get_category_popularity_score(candidate_category):
    return float(category_popularity_counter.get(candidate_category, 0))


print("Feature helper functions ready")

<!-- AUTO_CELL_EXPLANATION -->
### Cell 12: def get_top_copurchase_candidates(context_item_ids, top_n=30):
# Candidate Generation (Multi-Source) – Code Overview

## 1. Explanation (Precise)
- Generates candidate items from multiple sources:
  - Co-purchase (item–item relationships)
  - Context-based popularity
  - User history
  - NGCF model output
  - Allowed category filtering
- Merges all candidates and removes duplicates.
- Applies filtering rules:
  - Must belong to allowed categories
  - Optionally exclude same-category items
  - Removes items already in the basket
- Produces a **final candidate pool** for ranking.
## 2. Possible Output
```bash
Optimized candidate generation functions ready

In [ ]:
def get_top_copurchase_candidates(context_item_ids, top_n=30):
    score_counter = Counter()
    
    for context_item_id in context_item_ids:
        related_counter = pair_to_related_items.get(int(context_item_id), Counter())
        score_counter.update(related_counter)
    
    return [item_id for item_id, _ in score_counter.most_common(top_n)]


def get_top_context_candidates(context_key, top_n=30):
    score_counter = context_to_items.get(context_key, Counter())
    return [item_id for item_id, _ in score_counter.most_common(top_n)]


def get_top_user_history_candidates(customer_id, top_n=30):
    score_counter = user_to_items.get(int(customer_id), Counter())
    return [item_id for item_id, _ in score_counter.most_common(top_n)]


def get_allowed_category_candidates(
    allowed_categories,
    input_categories,
    strict_different_category_only,
    top_n=80
):
    temp = filter_catalog_by_allowed_categories(
        allowed_categories=allowed_categories,
        input_categories=input_categories,
        strict_different_category_only=strict_different_category_only
    )
    
    if temp.empty:
        return []
    
    temp = temp.copy()
    
    temp["categoryPopularity"] = temp["category"].map(
        lambda c: category_popularity_counter.get(c, 0)
    )
    
    if "totalRowsSeen" not in temp.columns:
        temp["totalRowsSeen"] = 1
    
    temp["totalRowsSeen"] = pd.to_numeric(
        temp["totalRowsSeen"],
        errors="coerce"
    ).fillna(1)
    
    temp = temp.sort_values(
        ["categoryPopularity", "totalRowsSeen"],
        ascending=[False, False]
    )
    
    return temp["itemId"].astype(int).head(top_n).tolist()


def build_candidate_pool(
    customer_id,
    context_item_ids,
    context_key,
    allowed_categories,
    input_categories,
    strict_different_category_only
):
    candidate_items = []
    
    candidate_items.extend(get_top_copurchase_candidates(context_item_ids, top_n=30))
    candidate_items.extend(get_top_context_candidates(context_key, top_n=30))
    candidate_items.extend(get_top_user_history_candidates(customer_id, top_n=30))
    candidate_items.extend(get_top_ngcf_candidates(customer_id, top_n=80))
    
    candidate_items.extend(
        get_allowed_category_candidates(
            allowed_categories=allowed_categories,
            input_categories=input_categories,
            strict_different_category_only=strict_different_category_only,
            top_n=80
        )
    )
    
    candidate_items = list(dict.fromkeys([int(x) for x in candidate_items]))
    
    context_item_set = set([int(x) for x in context_item_ids])
    allowed_category_set = set(allowed_categories)
    input_category_set = set(input_categories)
    
    filtered_items = []
    
    for item_id in candidate_items:
        category = item_to_category.get(int(item_id), "")
        
        if category not in allowed_category_set:
            continue
        
        if strict_different_category_only and category in input_category_set:
            continue
        
        if item_id in context_item_set:
            continue
        
        filtered_items.append(item_id)
    
    return filtered_items


print("Optimized candidate generation functions ready")

<!-- AUTO_CELL_EXPLANATION -->
### Cell 13: def build_ranker_row(
# Candidate Generation (Multi-Source) – Code Overview

## 1. Explanation (Precise)
- Generates candidate items from multiple sources:
  - Co-purchase (item–item relationships)
  - Context-based popularity
  - User history
  - NGCF model output
  - Allowed category filtering
- Merges all candidates and removes duplicates.
- Applies filtering rules:
  - Must belong to allowed categories
  - Optionally exclude same-category items
  - Removes items already in the basket
- Produces a **final candidate pool** for ranking.

## 2. Possible Output

```bash
Optimized candidate generation functions ready


In [ ]:
def build_ranker_row(
    group_id,
    basket_id,
    customer_id,
    context_item_ids,
    candidate_item_id,
    label,
    season,
    season_label,
    is_holiday,
    is_festival,
    time_slot,
    time_slot_label,
    month_part_label,
    week_of_month,
    allowed_categories,
    input_categories,
    rule_mode,
    allowed_category_weight_map
):
    candidate_item_id = int(candidate_item_id)
    candidate_category = item_to_category.get(candidate_item_id, "")
    candidate_name = item_to_name.get(candidate_item_id, "")
    candidate_family = get_category_family(candidate_category)

    input_families = [
        get_category_family(category)
        for category in input_categories
    ]
    input_families = list(dict.fromkeys(input_families))

    candidate_family_in_input = 1 if candidate_family in input_families else 0
    mixed_basket_flag = 1 if len(input_families) > 1 else 0
    input_family_count = len(input_families)

    context_key = (
        season,
        int(is_holiday),
        int(is_festival),
        time_slot,
        int(week_of_month)
    )

    context_vector = build_context_embedding_from_items(context_item_ids)

    if candidate_category in allowed_categories:
        allowed_category_rank = allowed_categories.index(candidate_category) + 1
    else:
        allowed_category_rank = 999

    return {
        "group_id": int(group_id),
        "basket_id": int(basket_id),
        "customerId": int(customer_id),
        "candidate_item_id": candidate_item_id,
        "candidate_item_name": candidate_name,
        "candidate_category": candidate_category,
        "candidate_family": candidate_family,
        "basket_size": int(len(context_item_ids)),
        "item_cooccurrence_score": get_item_cooccurrence_score(candidate_item_id, context_item_ids),
        "category_affinity_score": get_category_affinity_score(customer_id, candidate_category),
        "context_popularity_score": get_context_popularity_score(context_key, candidate_item_id),
        "customer_history_score": get_customer_history_score(customer_id, candidate_item_id),
        "embedding_similarity_score": get_embedding_similarity_score(context_vector, candidate_item_id),
        "candidate_category_popularity": get_category_popularity_score(candidate_category),
        "allowed_category_rank": int(allowed_category_rank),
        "candidate_category_allowed": 1 if candidate_category in allowed_categories else 0,
        "same_category_as_context": 1 if candidate_category in input_categories else 0,
        "category_rule_weight": float(allowed_category_weight_map.get(candidate_category, 0.0)),
        "candidate_family_in_input": int(candidate_family_in_input),
        "mixed_basket_flag": int(mixed_basket_flag),
        "input_family_count": int(input_family_count),
        "ngcf_score": get_ngcf_score(customer_id, candidate_item_id),
        "ngcf_rank": get_ngcf_rank(customer_id, candidate_item_id),
        "season": season,
        "seasonLabel": int(season_label),
        "isHoliday": int(is_holiday),
        "isFestival": int(is_festival),
        "timeSlot": time_slot,
        "timeSlotLabel": int(time_slot_label),
        "monthPartLabel": int(month_part_label),
        "weekOfMonth": int(week_of_month),
        "rule_mode": rule_mode,
        "label": int(label)
    }


print("Dynamic ranker row builder ready")


<!-- AUTO_CELL_EXPLANATION -->
### Cell 14: MAX_BASKETS = None
# Ranker Dataset Generation (Training Data) – Code Overview

## 1. Explanation (Precise)
- Builds training data for a **learning-to-rank model**.
- For each basket:
  - Treats each item as a **positive sample (label = 1)**.
  - Generates **negative samples (label = 0)** from candidate pool.
- Applies constraints:
  - Basket size filtering (min/max)
  - Category rule filtering
- Uses:
  - Dynamic category rules
  - Candidate generation logic
- Groups samples by `group_id` (ranking group).
- Outputs a structured dataset for training (e.g., XGBoost ranker).

## 2. Possible Output

```bash
Processed baskets: 1000
Processed baskets: 2000
...

Ranker dataset shape: (400000, 30)
Unique groups: 50000

Label distribution:
1    50000
0    350000

Mixed basket flag distribution:
1    30000
0    20000

Candidate family distribution:
cooking        120000
snack           80000
personal_care   50000
...


1. Explanation (Precise)
- Generates training data for a **learning-to-rank recommendation model**.
- Iterates through each basket and:
  - Treats each item as a **positive sample (label = 1)**.
  - Generates multiple **negative samples (label = 0)** from candidate pool.
- Applies constraints:
  - Basket size filtering (min/max limits)
  - Rule-based category filtering
- Uses:
  - Dynamic category rules
  - Multi-source candidate generation
- Groups samples using `group_id` for ranking tasks.
- Produces a dataset suitable for models like XGBoost ranker.

```bash
Processed baskets: 1000
Processed baskets: 2000
...

Ranker dataset shape: (400000, 30)
Unique groups: 50000

Label distribution:
1    50000
0    350000

Mixed basket flag distribution:
1    30000
0    20000

Candidate family distribution:
cooking        120000
snack           80000
personal_care   50000
...

In [ ]:
MAX_BASKETS = None
NEGATIVE_PER_POSITIVE = 8
MIN_BASKET_SIZE = 2
MAX_BASKET_SIZE = 20

training_rows = []
group_counter = 0

source_df = basket_df.copy()

if MAX_BASKETS is not None:
    source_df = source_df.head(MAX_BASKETS).copy()

for row_number, row in enumerate(source_df.itertuples(index=False), start=1):
    if row_number % 1000 == 0:
        print("Processed baskets:", row_number)

    basket_items = list(dict.fromkeys([int(x) for x in row.itemIdSet_list]))

    if len(basket_items) < MIN_BASKET_SIZE:
        continue

    if len(basket_items) > MAX_BASKET_SIZE:
        continue

    customer_id = int(row.customerId)
    basket_id = int(row.basket_id)

    season = row.season
    season_label = safe_int(row.seasonLabel)
    is_holiday = safe_int(row.isHoliday)
    is_festival = safe_int(row.isFestival)
    time_slot = row.timeSlot
    time_slot_label = safe_int(row.timeSlotLabel)
    month_part_label = safe_int(row.monthPartLabel)
    week_of_month = safe_int(row.weekOfMonth)

    for target_item in basket_items:
        target_item = int(target_item)
        target_category = item_to_category.get(target_item, "")

        context_item_ids = [int(x) for x in basket_items if int(x) != target_item]
        context_categories = [item_to_category.get(int(x), "") for x in context_item_ids]
        context_categories = [c for c in context_categories if c]
        context_categories = list(dict.fromkeys(context_categories))

        (
            allowed_categories,
            strict_different_category_only,
            rule_mode,
            allowed_category_weight_map
        ) = build_allowed_categories_from_context(
            context_categories,
            time_slot=time_slot
        )

        if not allowed_categories:
            continue

        if target_category not in allowed_categories:
            continue

        if strict_different_category_only and target_category in context_categories:
            continue

        context_key = (
            season,
            int(is_holiday),
            int(is_festival),
            time_slot,
            int(week_of_month)
        )

        candidate_pool = build_candidate_pool(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            context_key=context_key,
            allowed_categories=allowed_categories,
            input_categories=context_categories,
            strict_different_category_only=strict_different_category_only
        )

        if target_item not in candidate_pool:
            candidate_pool.append(target_item)

        context_item_set = set(context_item_ids)
        candidate_pool = [x for x in candidate_pool if x not in context_item_set]
        candidate_pool = list(dict.fromkeys(candidate_pool))

        negative_candidates = [x for x in candidate_pool if x != target_item]

        if len(negative_candidates) == 0:
            continue

        if len(negative_candidates) > NEGATIVE_PER_POSITIVE:
            negative_candidates = random.sample(negative_candidates, NEGATIVE_PER_POSITIVE)

        group_counter += 1
        group_id = group_counter

        positive_row = build_ranker_row(
            group_id=group_id,
            basket_id=basket_id,
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            candidate_item_id=target_item,
            label=1,
            season=season,
            season_label=season_label,
            is_holiday=is_holiday,
            is_festival=is_festival,
            time_slot=time_slot,
            time_slot_label=time_slot_label,
            month_part_label=month_part_label,
            week_of_month=week_of_month,
            allowed_categories=allowed_categories,
            input_categories=context_categories,
            rule_mode=rule_mode,
            allowed_category_weight_map=allowed_category_weight_map
        )

        training_rows.append(positive_row)

        for negative_item in negative_candidates:
            negative_row = build_ranker_row(
                group_id=group_id,
                basket_id=basket_id,
                customer_id=customer_id,
                context_item_ids=context_item_ids,
                candidate_item_id=negative_item,
                label=0,
                season=season,
                season_label=season_label,
                is_holiday=is_holiday,
                is_festival=is_festival,
                time_slot=time_slot,
                time_slot_label=time_slot_label,
                month_part_label=month_part_label,
                week_of_month=week_of_month,
                allowed_categories=allowed_categories,
                input_categories=context_categories,
                rule_mode=rule_mode,
                allowed_category_weight_map=allowed_category_weight_map
            )

            training_rows.append(negative_row)

ranker_df = pd.DataFrame(training_rows)

if ranker_df.empty:
    raise ValueError("Ranker dataset is empty. Check category rules, basket size, and candidate generation.")

print("Ranker dataset shape:", ranker_df.shape)
print("Unique groups:", ranker_df["group_id"].nunique())
print("Label distribution:")
print(ranker_df["label"].value_counts())

print("Mixed basket flag distribution:")
print(ranker_df["mixed_basket_flag"].value_counts())

print("Candidate family distribution:")
print(ranker_df["candidate_family"].value_counts().head(20))

display(ranker_df.head(20))


<!-- AUTO_CELL_EXPLANATION -->
### Cell 15: ranker_df.to_csv(RANKER_DATASET_FILE, index=False)
# Ranker Dataset Saving – Code Overview

## 1. Explanation (Precise)
- Saves the generated ranking dataset to a CSV file.
- Prints summary statistics:
  - Total rows
  - Number of ranking groups
  - Count of positive and negative samples

## 2. Possible Output

```bash
Saved: C:\D drive\item_recommendation_model_create\data\ranker_training_output\ranker_training_dataset.csv
Rows: 400000
Groups: 50000
Positive rows: 50000
Negative rows: 350000


In [ ]:
ranker_df.to_csv(RANKER_DATASET_FILE, index=False)

print("Saved:", RANKER_DATASET_FILE)
print("Rows:", len(ranker_df))
print("Groups:", ranker_df["group_id"].nunique())
print("Positive rows:", int((ranker_df["label"] == 1).sum()))
print("Negative rows:", int((ranker_df["label"] == 0).sum()))

Ranker Dataset Loading & Inspection – Code Overview

## 1. Explanation (Precise)
- Loads the saved ranker dataset from CSV.
- Prints dataset shape and column names.
- Displays first few rows for quick inspection.
- Used to verify dataset before model training.

## 2. Possible Output

```bash
Ranker dataset shape: (400000, 30)

Columns:
['group_id', 'basket_id', 'customer_id', 'candidate_item_id',
 'label', 'season', 'timeSlot', 'mixed_basket_flag', ...]

In [ ]:
ranker_df = pd.read_csv(RANKER_DATASET_FILE)

print("Ranker dataset shape:", ranker_df.shape)
print("Columns:")
print(ranker_df.columns.tolist())

display(ranker_df.head())

<!-- AUTO_CELL_EXPLANATION -->
### Cell 17: required_training_cols = [
# Feature Preparation for Model Training – Code Overview

## 1. Explanation (Precise)
- Validates required training columns.
- Handles missing values for categorical fields.
- Splits features into:
  - **Numeric features**
  - **Categorical features (one-hot encoded)**
- Cleans numeric data (removes NaN/inf).
- Combines all features into final matrix `X_all`.
- Extracts:
  - Labels (`y_all`)
  - Group IDs (`group_ids`)
- Prepares dataset for model training (e.g., XGBoost ranker).

## 2. Possible Output
```bash
Feature count: 85
X shape: (400000, 85)

y distribution:
0    350000
1     50000

New dynamic features included:
['candidate_family',
 'category_rule_weight',
 'candidate_family_in_input',
 'mixed_basket_flag',
 'input_family_count']

In [ ]:
required_training_cols = [
    "group_id",
    "candidate_item_id",
    "candidate_category",
    "candidate_family",
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "candidate_category_popularity",
    "allowed_category_rank",
    "candidate_category_allowed",
    "same_category_as_context",
    "category_rule_weight",
    "candidate_family_in_input",
    "mixed_basket_flag",
    "input_family_count",
    "ngcf_score",
    "ngcf_rank",
    "seasonLabel",
    "isHoliday",
    "isFestival",
    "timeSlotLabel",
    "monthPartLabel",
    "weekOfMonth",
    "label"
]

missing_training_cols = [col for col in required_training_cols if col not in ranker_df.columns]

if missing_training_cols:
    raise ValueError(f"Missing training columns: {missing_training_cols}")

ranker_df["candidate_category"] = ranker_df["candidate_category"].fillna("Unknown").astype(str)
ranker_df["candidate_family"] = ranker_df["candidate_family"].fillna("other").astype(str)
ranker_df["season"] = ranker_df["season"].fillna("Unknown").astype(str)
ranker_df["timeSlot"] = ranker_df["timeSlot"].fillna("Unknown").astype(str)
ranker_df["rule_mode"] = ranker_df["rule_mode"].fillna("Unknown").astype(str)

numeric_feature_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "candidate_category_popularity",
    "allowed_category_rank",
    "candidate_category_allowed",
    "same_category_as_context",
    "category_rule_weight",
    "candidate_family_in_input",
    "mixed_basket_flag",
    "input_family_count",
    "ngcf_score",
    "ngcf_rank",
    "seasonLabel",
    "isHoliday",
    "isFestival",
    "timeSlotLabel",
    "monthPartLabel",
    "weekOfMonth"
]

categorical_feature_cols = [
    "candidate_category",
    "candidate_family",
    "season",
    "timeSlot",
    "rule_mode"
]

model_df = ranker_df.copy()

X_categorical = pd.get_dummies(
    model_df[categorical_feature_cols],
    prefix=categorical_feature_cols,
    dtype=np.int8
)

X_numeric = model_df[numeric_feature_cols].copy()
X_numeric = X_numeric.replace([np.inf, -np.inf], 0).fillna(0)

X_all = pd.concat([X_numeric, X_categorical], axis=1)
y_all = model_df["label"].astype(int)
group_ids = model_df["group_id"].astype(int)

feature_cols = X_all.columns.tolist()

print("Feature count:", len(feature_cols))
print("X shape:", X_all.shape)
print("y distribution:")
print(y_all.value_counts())

print("New dynamic features included:")
print([
    "candidate_family",
    "category_rule_weight",
    "candidate_family_in_input",
    "mixed_basket_flag",
    "input_family_count"
])

display(X_all.head())


<!-- AUTO_CELL_EXPLANATION -->
### Cell 18: unique_groups = model_df["group_id"].drop_duplicates().tolist()
# Train-Test Split (Group-Based) – Code Overview

## 1. Explanation (Precise)
- Splits dataset into **train (80%)** and **test (20%)** using `group_id`.
- Ensures all rows from the same group stay in the same split.
- Randomly shuffles groups for unbiased splitting.
- Creates:
  - `X_train`, `y_train`, `qid_train`
  - `X_test`, `y_test`, `qid_test`
- Maintains ranking structure for model training.

## 2. Possible Output
```bash
Train rows: 320000
Test rows: 80000

Train groups: 40000
Test groups: 10000

Train label distribution:
0    280000
1     40000

Test label distribution:
0    70000
1    10000

In [ ]:
unique_groups = model_df["group_id"].drop_duplicates().tolist()

random.seed(42)
random.shuffle(unique_groups)

train_size = int(len(unique_groups) * 0.80)

train_groups = set(unique_groups[:train_size])
test_groups = set(unique_groups[train_size:])

train_mask = model_df["group_id"].isin(train_groups)
test_mask = model_df["group_id"].isin(test_groups)

X_train = X_all.loc[train_mask].copy()
y_train = y_all.loc[train_mask].copy()
qid_train = group_ids.loc[train_mask].copy()

X_test = X_all.loc[test_mask].copy()
y_test = y_all.loc[test_mask].copy()
qid_test = group_ids.loc[test_mask].copy()

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Train groups:", qid_train.nunique())
print("Test groups:", qid_test.nunique())

print("\nTrain label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

<!-- AUTO_CELL_EXPLANATION -->
### Cell 19: def sort_by_group(X, y, qid):
# Group Sorting for Ranking Model – Code Overview

## 1. Explanation (Precise)
- Sorts dataset by `group_id` (qid) for ranking model compatibility.
- Combines features, labels, and group IDs temporarily.
- Reorders data so all rows of a group are consecutive.
- Extracts:
  - Sorted features (`X`)
  - Labels (`y`)
  - Group IDs (`qid`)
- Computes `group_sizes` → number of samples per group (required for ranking models like XGBoost).


## 2. Possible Output
```bash
Train sorted shape: (320000, 85)
Test sorted shape: (80000, 85)

Train group count: 40000
Test group count: 10000

First train group sizes: [9, 9, 9, 9, 9, 9, 9, 9, 9, 9]

In [ ]:
def sort_by_group(X, y, qid):
    temp = X.copy()
    temp["label"] = y.values
    temp["qid"] = qid.values
    
    temp = temp.sort_values("qid").reset_index(drop=True)
    
    sorted_qid = temp["qid"].copy()
    sorted_y = temp["label"].copy()
    sorted_X = temp.drop(columns=["label", "qid"]).copy()
    
    group_sizes = sorted_qid.value_counts(sort=False).sort_index().tolist()
    
    return sorted_X, sorted_y, sorted_qid, group_sizes


X_train_sorted, y_train_sorted, qid_train_sorted, train_group_sizes = sort_by_group(
    X_train,
    y_train,
    qid_train
)

X_test_sorted, y_test_sorted, qid_test_sorted, test_group_sizes = sort_by_group(
    X_test,
    y_test,
    qid_test
)

print("Train sorted shape:", X_train_sorted.shape)
print("Test sorted shape:", X_test_sorted.shape)
print("Train group count:", len(train_group_sizes))
print("Test group count:", len(test_group_sizes))
print("First train group sizes:", train_group_sizes[:10])

<!-- AUTO_CELL_EXPLANATION -->
### Cell 20: ranker_model = xgb.XGBRanker(
# XGBoost Ranker Training – Code Overview

## 1. Explanation (Precise)
- Initializes an `XGBRanker` model for **learning-to-rank** tasks.
- Uses **NDCG@10** as the evaluation metric.
- Configures hyperparameters (depth, learning rate, regularization, etc.).
- Trains the model using:
  - Sorted training data
  - Group sizes (for ranking structure)
- Evaluates performance on test data during training.

## 2. Possible Output
```bash
[0]     validation_0-ndcg@10:0.51234
[50]    validation_0-ndcg@10:0.62345
[100]   validation_0-ndcg@10:0.67120
...
[699]   validation_0-ndcg@10:0.74580


In [ ]:
ranker_model = xgb.XGBRanker(
    objective="rank:ndcg",
    eval_metric="ndcg@10",
    n_estimators=700,
    learning_rate=0.035,
    max_depth=5,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_lambda=2.0,
    reg_alpha=0.15,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

ranker_model.fit(
    X_train_sorted,
    y_train_sorted,
    group=train_group_sizes,
    eval_set=[(X_test_sorted, y_test_sorted)],
    eval_group=[test_group_sizes],
    verbose=True
)

print("XGBoost Ranker training completed")

<!-- AUTO_CELL_EXPLANATION -->
### Cell 21: test_scored_df = model_df.loc[test_mask].copy()
# Test Prediction & Scoring – Code Overview

## 1. Explanation (Precise)
- Selects test portion of dataset using `test_mask`.
- Sorts data by `group_id` to align with model input.
- Uses trained XGBoost ranker to generate **prediction scores**.
- Adds `pred_score` column to dataset.
- Displays key columns for evaluation and inspection.
## 2. Possible Output
```bash
Scored test shape: (80000, 35)


In [ ]:
test_scored_df = model_df.loc[test_mask].copy()
test_scored_df = test_scored_df.sort_values("group_id").reset_index(drop=True)

test_scored_df["pred_score"] = ranker_model.predict(X_test_sorted)

print("Scored test shape:", test_scored_df.shape)

display(
    test_scored_df[
        [
            "group_id",
            "candidate_item_id",
            "candidate_item_name",
            "candidate_category",
            "label",
            "ngcf_score",
            "ngcf_rank",
            "pred_score"
        ]
    ].head(20)
)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 22: def precision_at_k(labels, scores, k):
# Ranking Evaluation Metrics – Code Overview

## 1. Explanation (Precise)
- Implements evaluation metrics for ranking models:
  - **Precision@K** → relevance in top-K results
  - **Recall@K** → coverage of relevant items
  - **NDCG@K** → ranking quality (position-aware)
  - **MRR@K** → first relevant item position
  - **AUC** → overall classification quality
- Evaluates model **per group (query)** and computes average summary.
- Outputs:
  - `metric_df` → per-group metrics
  - `metric_summary` → overall performance

## 2. Possible Output

```bash
Metric summary:
precision_at_3 : 0.62
recall_at_3 : 0.41
ndcg_at_3 : 0.68
mrr_at_3 : 0.72
precision_at_5 : 0.58
recall_at_5 : 0.55
ndcg_at_5 : 0.70
mrr_at_5 : 0.74
precision_at_10 : 0.50
recall_at_10 : 0.72
ndcg_at_10 : 0.75
mrr_at_10 : 0.76
auc : 0.88


In [ ]:
def precision_at_k(labels, scores, k):
    order = np.argsort(scores)[::-1][:k]
    top_labels = np.array(labels)[order]
    
    return float(np.sum(top_labels) / k)


def recall_at_k(labels, scores, k):
    labels = np.array(labels)
    total_positive = np.sum(labels)
    
    if total_positive == 0:
        return 0.0
    
    order = np.argsort(scores)[::-1][:k]
    top_labels = labels[order]
    
    return float(np.sum(top_labels) / total_positive)


def ndcg_at_k(labels, scores, k):
    labels = np.array(labels)
    scores = np.array(scores)
    
    order = np.argsort(scores)[::-1][:k]
    ranked_labels = labels[order]
    
    dcg = 0.0
    
    for i, rel in enumerate(ranked_labels, start=1):
        dcg += (2 ** rel - 1) / np.log2(i + 1)
    
    ideal_labels = np.sort(labels)[::-1][:k]
    
    idcg = 0.0
    
    for i, rel in enumerate(ideal_labels, start=1):
        idcg += (2 ** rel - 1) / np.log2(i + 1)
    
    if idcg == 0:
        return 0.0
    
    return float(dcg / idcg)


def mrr_at_k(labels, scores, k):
    labels = np.array(labels)
    scores = np.array(scores)
    
    order = np.argsort(scores)[::-1][:k]
    ranked_labels = labels[order]
    
    for i, rel in enumerate(ranked_labels, start=1):
        if rel == 1:
            return float(1 / i)
    
    return 0.0


def evaluate_ranker(scored_df, k_values=[3, 5, 10]):
    rows = []
    
    for group_id, group in scored_df.groupby("group_id"):
        labels = group["label"].values
        scores = group["pred_score"].values
        
        row = {
            "group_id": group_id,
            "group_size": len(group),
            "positive_count": int(np.sum(labels))
        }
        
        for k in k_values:
            row[f"precision_at_{k}"] = precision_at_k(labels, scores, k)
            row[f"recall_at_{k}"] = recall_at_k(labels, scores, k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(labels, scores, k)
            row[f"mrr_at_{k}"] = mrr_at_k(labels, scores, k)
        
        rows.append(row)
    
    metric_df = pd.DataFrame(rows)
    
    summary = {}
    
    for k in k_values:
        summary[f"precision_at_{k}"] = float(metric_df[f"precision_at_{k}"].mean())
        summary[f"recall_at_{k}"] = float(metric_df[f"recall_at_{k}"].mean())
        summary[f"ndcg_at_{k}"] = float(metric_df[f"ndcg_at_{k}"].mean())
        summary[f"mrr_at_{k}"] = float(metric_df[f"mrr_at_{k}"].mean())
    
    try:
        summary["auc"] = float(roc_auc_score(scored_df["label"], scored_df["pred_score"]))
    except Exception:
        summary["auc"] = None
    
    return metric_df, summary


metric_df, metric_summary = evaluate_ranker(test_scored_df, k_values=[3, 5, 10])

print("Metric summary:")
for key, value in metric_summary.items():
    print(key, ":", value)

display(metric_df.head())

<!-- AUTO_CELL_EXPLANATION -->
### Cell 23: feature_importance_df = pd.DataFrame({
**Explanation:** This cell writes processed outputs/artifacts to disk from the ranker training and XGBoost pipeline.

**Possible Output:** May print save path/logs, or finish silently after writing files.


In [ ]:
feature_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": ranker_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(30))

feature_importance_df.to_csv(FEATURE_IMPORTANCE_FILE, index=False)

print("Saved:", FEATURE_IMPORTANCE_FILE)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 24: ranker_model.save_model(MODEL_JSON_FILE)
**Explanation:** This cell writes processed outputs/artifacts to disk from the ranker training and XGBoost pipeline.

**Possible Output:** May print save path/logs, or finish silently after writing files.


In [ ]:
ranker_model.save_model(MODEL_JSON_FILE)

with open(MODEL_PICKLE_FILE, "wb") as f:
    pickle.dump(ranker_model, f)

feature_artifact = {
    "numeric_feature_cols": numeric_feature_cols,
    "categorical_feature_cols": categorical_feature_cols,
    "final_feature_cols": feature_cols
}

with open(FEATURE_COLUMNS_FILE, "w", encoding="utf-8") as f:
    json.dump(feature_artifact, f, ensure_ascii=False, indent=4)

test_scored_df.to_csv(TEST_PREDICTION_FILE, index=False)

training_summary = {
    "ranker_dataset_file": str(RANKER_DATASET_FILE),
    "model_json_file": str(MODEL_JSON_FILE),
    "model_pickle_file": str(MODEL_PICKLE_FILE),
    "feature_columns_file": str(FEATURE_COLUMNS_FILE),
    "ngcf_candidate_file": str(NGCF_TOP_CANDIDATES_FILE),
    "total_rows": int(len(ranker_df)),
    "total_groups": int(ranker_df["group_id"].nunique()),
    "train_rows": int(X_train_sorted.shape[0]),
    "test_rows": int(X_test_sorted.shape[0]),
    "train_groups": int(len(train_group_sizes)),
    "test_groups": int(len(test_group_sizes)),
    "feature_count": int(len(feature_cols)),
    "negative_per_positive": int(NEGATIVE_PER_POSITIVE),
    "metrics": metric_summary,
    "model_params": ranker_model.get_params()
}

with open(TRAINING_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, ensure_ascii=False, indent=4)

print("Saved model and artifacts:")
print(MODEL_JSON_FILE)
print(MODEL_PICKLE_FILE)
print(FEATURE_COLUMNS_FILE)
print(TRAINING_SUMMARY_FILE)
print(TEST_PREDICTION_FILE)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 25: final_result_summary_df = pd.DataFrame([
**Explanation:** This cell writes processed outputs/artifacts to disk from the ranker training and XGBoost pipeline.

**Possible Output:** May print save path/logs, or finish silently after writing files.


In [ ]:
final_result_summary_df = pd.DataFrame([
    {
        "model": "XGBoost Ranker Dynamic Mixed Basket with NGCF Candidates",
        "objective": "rank:ndcg",
        "totalRows": int(len(ranker_df)),
        "totalGroups": int(ranker_df["group_id"].nunique()),
        "trainRows": int(X_train_sorted.shape[0]),
        "testRows": int(X_test_sorted.shape[0]),
        "featureCount": int(len(feature_cols)),
        "precisionAt3": metric_summary.get("precision_at_3"),
        "recallAt3": metric_summary.get("recall_at_3"),
        "ndcgAt3": metric_summary.get("ndcg_at_3"),
        "mrrAt3": metric_summary.get("mrr_at_3"),
        "precisionAt5": metric_summary.get("precision_at_5"),
        "recallAt5": metric_summary.get("recall_at_5"),
        "ndcgAt5": metric_summary.get("ndcg_at_5"),
        "mrrAt5": metric_summary.get("mrr_at_5"),
        "precisionAt10": metric_summary.get("precision_at_10"),
        "recallAt10": metric_summary.get("recall_at_10"),
        "ndcgAt10": metric_summary.get("ndcg_at_10"),
        "mrrAt10": metric_summary.get("mrr_at_10"),
        "auc": metric_summary.get("auc")
    }
])

final_result_summary_df.to_csv(FINAL_RESULT_FILE, index=False)

print("Saved:", FINAL_RESULT_FILE)

display(final_result_summary_df)


<!-- AUTO_CELL_EXPLANATION -->
### Cell 26: sample_group_id = test_scored_df["group_id"].iloc[0]
**Explanation:** This cell inspects data/model outputs for validation in the ranker training and XGBoost pipeline.

**Possible Output:** May run silently with variable/state updates and no rendered output.


In [ ]:
sample_group_id = test_scored_df["group_id"].iloc[0]

sample_recommendation = (
    test_scored_df[test_scored_df["group_id"] == sample_group_id]
    .sort_values("pred_score", ascending=False)
    [
        [
            "group_id",
            "customerId",
            "candidate_item_id",
            "candidate_item_name",
            "candidate_category",
            "label",
            "ngcf_score",
            "ngcf_rank",
            "pred_score",
            "rule_mode"
        ]
    ]
)

print("Sample group id:", sample_group_id)
display(sample_recommendation)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 27: loaded_model = xgb.XGBRanker()
**Explanation:** This cell trains or fits model components in the ranker training and XGBoost pipeline.

**Possible Output:** May run silently with variable/state updates and no rendered output.


In [ ]:
loaded_model = xgb.XGBRanker()
loaded_model.load_model(MODEL_JSON_FILE)

sample_pred = loaded_model.predict(X_test_sorted.head(10))

print("Reloaded model prediction sample:")
print(sample_pred)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 28: output_files = [
**Explanation:** This cell trains or fits model components in the ranker training and XGBoost pipeline.

**Possible Output:** May run silently with variable/state updates and no rendered output.


In [ ]:
output_files = [
    RANKER_DATASET_FILE,
    MODEL_JSON_FILE,
    MODEL_PICKLE_FILE,
    FEATURE_COLUMNS_FILE,
    TRAINING_SUMMARY_FILE,
    FEATURE_IMPORTANCE_FILE,
    TEST_PREDICTION_FILE,
    FINAL_RESULT_FILE
]

for file_path in output_files:
    print(file_path.name, "exists:", file_path.exists(), "path:", file_path)

<!-- AUTO_CELL_EXPLANATION -->
### Cell 29: Code cell execution step
**Explanation:** This cell is a placeholder in the ranker training and XGBoost pipeline.

**Possible Output:** May run silently with variable/state updates and no rendered output.
